# Train Laya for Researchly on a free Kaggle GPU

Trains both S5 critics (Task B: sentence roles; Task A: is this flag useful?), evaluates them,
calibrates them, exports INT8 ONNX and zips everything into `laya-outputs.zip`.

Before you start (details in `ml/critics/laya/README.md` of the repository):

1. **Add data** (right panel) → *Your datasets* → the private dataset you made from `laya-kit.zip`.
2. **Settings** → *Accelerator* **GPU T4 x2** (one GPU is used) and *Internet* **on** (the base model
   and `laya` are downloaded).
3. **Save Version** → *Save & Run All (Commit)*. It runs in the background (a full run takes about
   1–2 hours on one T4; Kaggle allows 12). When it finishes, open the version's **Output** tab and download
   `laya-outputs.zip`.
4. On your Mac: `bash ml/critics/laya/train.sh --import ~/Downloads/laya-outputs.zip`

Nothing in this notebook sees your own labelled paragraphs: they never leave your Mac.

In [ ]:
# Settings
TASK = 'both'        # 'a', 'b' or 'both'
QUICK = False        # True: a few-minute smoke run on a small sample
KEEP_CHECKPOINTS = False  # True also keeps the PyTorch checkpoints (~850 MB each) in the output

In [ ]:
# 1. Find the kit and copy it somewhere writable
import glob, os, shutil
marks = glob.glob('/kaggle/input/**/laya-kit.json', recursive=True)
assert marks, 'Add the dataset made from laya-kit.zip (Add data -> Your datasets)'
KIT = '/kaggle/working/laya-kit'
RUN = '/kaggle/working/laya-run'
shutil.copytree(os.path.dirname(marks[0]), KIT, dirs_exist_ok=True)
print(open(os.path.join(KIT, 'laya-kit.json')).read())

In [ ]:
# 2. Install laya (Kaggle's own CUDA build of torch is kept) and check the GPU
!pip install -q -r /kaggle/working/laya-kit/ml/critics/laya/requirements-kaggle.txt
import torch
assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4 x2'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 3. The plan (data sizes, epochs, batch, time estimates)
quick = '--quick' if QUICK else ''
!cd {KIT} && python ml/critics/laya/pipeline.py plan --data data --task {TASK} --device cuda {quick}

In [ ]:
# 4. Train, evaluate, calibrate, export INT8 ONNX (Task B, then Task A).
#    Re-running this cell resumes: finished stages are skipped, and training
#    continues from the last finished epoch.
!cd {KIT} && python ml/critics/laya/pipeline.py run --data data --run {RUN} --task {TASK} --device cuda {quick}

In [ ]:
# 5. What came out (the gate runs on your Mac after --import)
!cd {KIT} && python ml/critics/laya/pipeline.py summary --run {RUN}

In [ ]:
# 6. Zip the outputs for download: the INT8 critics, reports and the data they were tested on
OUT = '/kaggle/working/laya-outputs'
shutil.rmtree(OUT, ignore_errors=True)
shutil.copytree(os.path.join(KIT, 'data'), os.path.join(OUT, 'data'))
skip = None if KEEP_CHECKPOINTS else shutil.ignore_patterns('ckpt')
for t in ('a', 'b'):
    if os.path.isdir(os.path.join(RUN, t)):
        shutil.copytree(os.path.join(RUN, t), os.path.join(OUT, t), ignore=skip)
shutil.make_archive('/kaggle/working/laya-outputs', 'zip', '/kaggle/working', 'laya-outputs')
# keep the Output tab small: only the zip remains
for d in (OUT, RUN, KIT):
    shutil.rmtree(d, ignore_errors=True)
print('Download /kaggle/working/laya-outputs.zip from the Output tab,')
print('then on your Mac: bash ml/critics/laya/train.sh --import ~/Downloads/laya-outputs.zip')
print(round(os.path.getsize('/kaggle/working/laya-outputs.zip') / 1e6), 'MB')